In [2]:
import pandas as pd
import numpy as np
import json
import os
import zipfile
import glob
import time
import datetime

# Importing BeautifulSoup and 
# it is in the bs4 module
from bs4 import * 
import codecs

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Using the already labelled files from Drive

In [5]:
df = pd.read_csv('/content/drive/MyDrive/Research stuff/Final Year Design Project/Survey and collected data/Total MindSeer_Data_Collection (Responses) - Form Responses 1.csv')
# df.head()

Scoring

In [6]:
data=np.array(df)
# data[0][4] #correct

In [7]:
row, col = data.shape
Class = []

for i in range(row):
    dif = [7,11,15,19]
    score = 0
    l = 0
    for j in range(4,25):
        if l<4 and j == dif[l]:
            if (data[i][j][0]) == 'R':
                score += 3
            elif (data[i][j][0]) == 'S':
                score += 2
            elif (data[i][j][0]) == 'O':
                score += 1
            elif (data[i][j][0]) == 'M':
                score += 0 
            l=l+1
        else:
            if (data[i][j][0]) == 'R':
                score += 0
            elif (data[i][j][0]) == 'S':
                score += 1
            elif (data[i][j][0]) == 'O':
                score += 2
            elif (data[i][j][0]) == 'M':
                score += 3

#       if(score<15): 
#           Class.append(0)
#       else:
#           # data = np.insert(data, 25, 1, axis=1)
#           Class.append(1)

    # Determining MDD (De Choudhury, "Predicting Depression via Social Media" paper's approach):
    if score < 24:
        Class.append(0)

    else:
        Class.append(1)

In [8]:
len(Class)

104

Finding number of participants in each class

In [9]:
negative_class = []
positive_class = []
for i in range(len(Class)):
    if Class[i] == 0:
        negative_class.append(i)
    else:
        positive_class.append(i)

print("Negative subjects = ", len(negative_class))
print("Positive subjects = ", len(positive_class))

Negative subjects =  60
Positive subjects =  44


Reading the files and storing them in dictionaries

In [ ]:
root_path = '/content/drive/MyDrive/Research stuff/Final Year Design Project/Survey and collected data/mindseer_data/'

participants = 0
for name in glob.glob(root_path + '*.zip'): 
    participants += 1

print(participants)

104


In [ ]:
#code snippet from https://stackoverflow.com/questions/50008296/facebook-json-badly-encoded

def parse_obj(obj):
    if isinstance(obj, str):
        return obj.encode('latin_1').decode('utf-8')

    if isinstance(obj, list):
        return [parse_obj(o) for o in obj]

    if isinstance(obj, dict):
        return {key: parse_obj(item) for key, item in obj.items()}

    return obj

In [ ]:
def data_insert(path, id, data_list):
    try:
        file_present = True
        f = open(path, 'r')
        try:
            raw_data = parse_obj(json.load(f))
            if (path[-6] == '1'):
                json_data = {}
                if type(raw_data) is dict:
                    json_data["posts"] = []
                    json_data["posts"].append(raw_data)
                else:
                    json_data["posts"] = raw_data
                    raw_data = json_data 
        except UnicodeDecodeError as error:
            print(error)
    except FileNotFoundError:
        file_present = False  
        raw_data = {}
    finally:
        #print(raw_data)
        raw_data['participant_id'] = id
        data_list.append(raw_data)
        if file_present:
            f.close()

In [ ]:
comments_data = []
groups_data = []
posts_data = []


for i in range(participants):
    comments_path = root_path + str(i) + '/comments/comments.json'
    data_insert(comments_path, i, comments_data)

    groups_path = root_path + str(i) + '/groups/your_posts_and_comments_in_groups.json'
    data_insert(groups_path, i, groups_data)

    posts_path = root_path + str(i) + '/posts/your_posts_1.json'
    data_insert(posts_path, i, posts_data)


In [ ]:
print(len(comments_data))
print(len(groups_data))
print(len(posts_data))

104
104
104


Keeping count of total posts made by users of each class

In [ ]:
negative_count = 0
positive_count = 0

Extracting relevant information

In [ ]:
negative_user_comments_count = 0
positive_user_comments_count = 0
comments_with_timestamps = []

for i in range(participants):
    try:
        for comment_details in comments_data[i]['comments']:
            #print(comment_details)
            try:
                for data in comment_details['data']:
                    comment = {}
                    comment['user_id'] = i
                    comment['text'] = data['comment']['comment']
                    comment['timestamp'] = data['comment']['timestamp']
                    # text += (data['comment']['comment'] + " ")
                    comments_with_timestamps.append(comment)
                    if i in negative_class:
                        negative_count += 1
                        negative_user_comments_count += 1
                    else:
                        positive_count += 1
                        positive_user_comments_count += 1

            except KeyError:
                comment = {}
                comment['user_id'] = i
                comment['text'] = None
                comment['timestamp'] = None
                comments_with_timestamps.append(comment)
                continue
    except KeyError:
        comment = {}
        comment['user_id'] = i
        comment['text'] = None
        comment['timestamp'] = None
        comments_with_timestamps.append(comment)
        continue
    
    # finally:
    #     #print(text)
    #     comment_texts.append(text)
    

In [ ]:
comments_with_timestamps[:5]

In [ ]:
#print('length of comments list =', len(comments_with_timestamps))
print("total comments made by positive class =", positive_user_comments_count)
print("total comments made by negative class =", negative_user_comments_count)

total comments made by positive class = 50202
total comments made by negative class = 34510


In [ ]:
negative_user_posts_count = 0
positive_user_posts_count = 0
posts_with_timestamps = []
# posts_texts = []

for i in range(participants):
    try:
        for post_details in posts_data[i]["posts"]:
            #print(post)
            post = {}
            post['user_id'] = i
            post['timestamp'] = post_details['timestamp']
            try:
                for data in post_details['data']:
                    #print(data)
                    if 'post' in data:
                        post['text'] = data['post']
                    else:
                        post['text'] = None
                    
                    posts_with_timestamps.append(post)

                    if i in negative_class:
                        negative_count += 1
                        negative_user_posts_count += 1
                    else:
                        positive_count += 1  
                        positive_user_posts_count += 1
            except KeyError:
                post['text'] = None
                posts_with_timestamps.append(post)
                continue                 
    except KeyError:
        post = {}
        post['user_id'] = i
        post['timestamp'] = None
        post['text'] = None
        posts_with_timestamps.append(post)
        continue

    # finally:
    #     posts_texts.append(text)



In [ ]:
posts_with_timestamps[:5]

In [ ]:
#print('length of posts list=', len(posts_with_timestamps))
print("total posts made by positive class =", positive_user_posts_count)
print("total posts made by negative class =", negative_user_posts_count)

total posts made by positive class = 23800
total posts made by negative class = 14396


In [ ]:
negative_user_group_comments_count = 0
positive_user_group_comments_count = 0
negative_user_group_posts_count = 0
positive_user_group_posts_count = 0
groups_posts_with_timestamps = []
groups_comments_with_timestamps = []

for i in range(participants):
    group_post = {}
    try:
        for posts_comments in groups_data[i]["group_posts"]["activity_log_data"]:
            # print(posts_comments) # data, title #all appears
            if 'data' in posts_comments:
                for data in posts_comments['data']:
                    #print(data)
                    if 'comment' in data:
                        #print('comment')
                        group_comment = {}
                        group_comment['user_id'] = i
                        group_comment['timestamp'] = data ['comment']['timestamp']
                        group_comment['text'] = data ['comment']['comment']
                        
                        if group_comment not in groups_comments_with_timestamps:
                            groups_comments_with_timestamps.append(group_comment)

                            if i in negative_class:
                                negative_count += 1
                                negative_user_group_comments_count += 1
                            else:
                                positive_count += 1
                                positive_user_group_comments_count += 1

                    elif 'post' in data:
                        group_post['user_id'] = i
                        #print("The text of the post is", data['post'])
                        group_post ['text'] = data['post']

                    elif 'update_timestamp' in data:
                        #print(data['update_timestamp'])
                        group_post['timestamp'] = data['update_timestamp']
                        groups_posts_with_timestamps.append(group_post)

                        if i in negative_class:
                            negative_count += 1
                            negative_user_group_posts_count += 1
                        else:
                            positive_count += 1
                            positive_user_group_posts_count += 1

                        group_post = {}

    except KeyError:
        group_post = {}
        group_comment = {}
        group_post['user_id'] = i
        group_comment['user_id'] = i
        group_post ['text'] = None
        group_post ['timestamp'] = None
        group_comment ['text'] = None
        group_comment ['timestamp'] = None 
        groups_posts_with_timestamps.append(group_post) 
        groups_comments_with_timestamps.append(group_comment)
        continue
            

In [ ]:
groups_comments_with_timestamps[:10]

In [ ]:
groups_posts_with_timestamps[:10]

In [ ]:
print('length of group comments list =', len(groups_comments_with_timestamps))
print("total group comments made by positive class =", positive_user_group_comments_count)
print("total group comments made by negative class =", negative_user_group_comments_count)
print()
print('length of group posts list=', len(groups_posts_with_timestamps))
print("total group posts made by positive class =", positive_user_group_posts_count)
print("total group posts made by negative class =", negative_user_group_posts_count)

length of group comments list = 16158
total group comments made by positive class = 9641
total group comments made by negative class = 6485

length of group posts list= 10026
total group posts made by positive class = 8216
total group posts made by negative class = 1778


In [ ]:
negative_count

57169

In [ ]:
positive_count

91859

In [ ]:
total_texts_in_both_classes = negative_count + positive_count
total_texts_in_both_classes

149028